# **Librerias**

In [1]:
import pandas as pd
import numpy as np
import Levenshtein

# **Carga dataset**

In [2]:
# Catálogo: 8747 libros con slug, title, authors, genres, synopsis y otros metadatos. 
# Algunos registros están contaminados; no se indica cuáles. 
catalogo = pd.read_json("Fuente de datos/catalogo_tp3.jsonl", lines=True)

# Muestra: 500 libros del mismo catálogo con la columna contaminado (1 o 0). 
# En los contaminados, la columna fragmento contiene el texto que se insertó. 
muestra = pd.read_csv("Fuente de datos/muestra_etiquetada.csv", keep_default_na=False)

In [3]:
# Renombramos columnas en español
reemplazo_columnas = {
    "source_url": "url",
    "slug": "slug",
    "title": "titulo",
    "authors": "autores",
    "genres": "generos",
    "seed_genre": "generos_semilla",
    "series": "series",
    "series_number": "numero_serie",
    "cover_url": "url_portada",
    "synopsis": "sinopsis",
    "synopsis_chars" : "cantidad_caracteres_sinopsis",
    "scraped_at": "fecha_scrapeo"}
catalogo.rename(columns=reemplazo_columnas, inplace=True)

# Añadimos una columna con la fecha de scrapeo en formato datetime para poder filtrar por fecha
catalogo["fecha_scrapeo"] = pd.to_datetime(catalogo["fecha_scrapeo"])

# Mostramos las primeras filas del catálogo
catalogo.head()

,url,slug,titulo,autores,generos,generos_semilla,series,numero_serie,url_portada,sinopsis,cantidad_caracteres_sinopsis,fecha_scrapeo
0,https://ww3.lectulandia.com/book/de-la-caverna...,de-la-caverna-al-cosmos,De la caverna al cosmos,[Eudald Carbonell],"[Arqueología, Divulgación, Ensayo]",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/Eudald%20C...,La Humanidad ha entrado en una nueva fase evol...,998,2026-10-17 23:11:56+00:00
1,https://ww3.lectulandia.com/book/quienes-somos...,quienes-somos-y-como-hemos-llegado-hasta-aqui,Quiénes somos y cómo hemos llegado hasta aquí,[David Reich],"[Arqueología, Biología, Ciencias naturales, En...",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/David%20Re...,La extracción de ADN de los huesos de nuestros...,1285,2026-10-17 23:11:58+00:00
2,https://ww3.lectulandia.com/book/teotihuacan/,teotihuacan,Teotihuacan,[Eduardo Matos Moctezuma],"[Arqueología, Arte, Ciencias sociales, Ensayo,...",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/Eduardo%20...,Teotihuacan es la primera gran ciudad del cent...,1070,2026-10-17 23:12:01+00:00
3,https://ww3.lectulandia.com/book/tiro-y-las-co...,tiro-y-las-colonias-fenicias-de-occidente,Tiro y las colonias fenicias de Occidente,[M.ª Eugenia Aubet],"[Arqueología, Economía, Ensayo, Historia, Polí...",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/M%20Eugeni...,Entre los siglos VIII y VI a. C. los fenicios ...,1875,2026-10-17 23:12:03+00:00
4,https://ww3.lectulandia.com/book/la-maldicion-...,la-maldicion-del-hombre-mono,La maldición del hombre mono,[Emiliano Bruner],"[Arqueología, Ensayo, Psicología, Sociología]",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/Emiliano%2...,Las raíces evolutivas del sufrimiento humano\n...,1613,2026-10-17 23:12:05+00:00


# **EDA** (Analisis Exploratorio de Datos)

In [4]:
# Eliminamos registros sin sinopsis y duplicados por identificador. 
catalogo.dropna(subset=["sinopsis"], inplace=True)

# Vemos ahora informacion sobre el catalogo
catalogo.info()

<class 'pandas.DataFrame'>
Index: 8746 entries, 0 to 8746
Data columns (total 12 columns):
 #   Column                        Non-Null Count  Dtype              
---  ------                        --------------  -----              
 0   url                           8746 non-null   str                
 1   slug                          8746 non-null   str                
 2   titulo                        8746 non-null   str                
 3   autores                       8746 non-null   object             
 4   generos                       8746 non-null   object             
 5   generos_semilla               8746 non-null   str                
 6   series                        2316 non-null   str                
 7   numero_serie                  2298 non-null   float64            
 8   url_portada                   8746 non-null   str                
 9   sinopsis                      8746 non-null   str                
 10  cantidad_caracteres_sinopsis  8746 non-null   int64 

## **Limpieza del texto**

In [5]:
catalogo[catalogo["sinopsis"].str.contains("\n", regex=True, na=False)]["sinopsis"]

0       La Humanidad ha entrado en una nueva fase evol...
1       La extracción de ADN de los huesos de nuestros...
2       Teotihuacan es la primera gran ciudad del cent...
3       Entre los siglos VIII y VI a. C. los fenicios ...
4       Las raíces evolutivas del sufrimiento humano\n...
                              ...                        
8733    El agente Gabriel Syme es reclutado por Scotla...
8734    En esta magnífica obra se cuenta la historia d...
8739    "El bolígrafo de gel verde es el retrato detal...
8743    Cuando una vieja escritora acostumbrada a ment...
8745    ¿Qué futuro tiene un país como España donde la...
Name: sinopsis, Length: 5247, dtype: str

In [6]:
def eliminar_espacios_extra(texto):
    if isinstance(texto, str):
        return " ".join(texto.split())
    return texto

def eliminar_acentos(texto):
    if isinstance(texto, str):
        return texto.translate(str.maketrans("áéíóúüñÁÉÍÓÚÜÑ", "aeiouunAEIOUUN"))
    return texto

def transformar_texto_minusculas(texto):
    if isinstance(texto, str):
        return texto.lower()
    return texto

def texto_normalizado(texto):
    texto = eliminar_espacios_extra(texto)
    texto = eliminar_acentos(texto)
    texto = transformar_texto_minusculas(texto)
    return texto

In [7]:
catalogo["sinopsis"].apply(texto_normalizado)

0       la humanidad ha entrado en una nueva fase evol...
1       la extraccion de adn de los huesos de nuestros...
2       teotihuacan es la primera gran ciudad del cent...
3       entre los siglos viii y vi a. c. los fenicios ...
4       las raices evolutivas del sufrimiento humano u...
                              ...                        
8742    janna y maudie poco o nada tienen en comun. ja...
8743    cuando una vieja escritora acostumbrada a ment...
8744    el tratado de la naturaleza humana es la cumbr...
8745    ¿que futuro tiene un pais como espana donde la...
8746    coincidiendo con el inicio de la ofensiva alem...
Name: sinopsis, Length: 8746, dtype: str

## **Duplicados**

In [8]:
print(f"Duplicados antes de eliminar: {catalogo.duplicated(subset=['sinopsis']).sum()}")
catalogo.drop_duplicates(subset=['sinopsis'], inplace=True)
print(f"Duplicados después de eliminar: {catalogo.duplicated(subset=['sinopsis']).sum()}")

Duplicados antes de eliminar: 98
Duplicados después de eliminar: 0


### **Similares**

In [ ]:
for autores in catalogo["autores"]:
    for autor in autores:
        sinopsis_autor = catalogo[catalogo["autores"].apply(lambda x: autor in x)]["sinopsis"]
        if len(sinopsis_autor) > 1:
            for i in range(len(sinopsis_autor)):
                for j in range(i + 1, len(sinopsis_autor)):
                    sinopsis_i = sinopsis_autor.iloc[i]
                    sinopsis_j = sinopsis_autor.iloc[j]
                    distancia = Levenshtein.distance(sinopsis_i, sinopsis_j)
                    if distancia < 10:
                        print("Sinopsis 1:", sinopsis_i)
                        print("Sinopsis 2:", sinopsis_j)
                        print("Distancia de Levenshtein:", distancia)
                        print()

Sinopsis 1: Recopilación de los artículos publicados en el suplemento dominical «XL Semanal» durante 2015. El contenido se ha obtenido a través de www.perezreverte.com
Sinopsis 2: Recopilación de los artículos publicados en el suplemento dominical «XL Semanal» durante 2013. El contenido se ha obtenido a través de www.perezreverte.com
Distancia de Levenshtein: 1

Sinopsis 1: De una forma clara y lúcida, el autor nos introduce en temas científicos como galaxias, átomos, relatividad o la evolución de las especies, y otros no tanto pero si de interés como el racismo, la sexualidad, el alma o las pseudociencias, tan de moda actualmente. Nos aproxima a estos temas de forma razonada, en ocasiones desde puntos de vista diferentes de los convencionales. El libro en si es una recopilación de artículos publicados por el autor en la revista The Magazine of Fantasy and Science Fiction, aunque actualizados ya que el primero lo escribió allá por el 1958.
Sinopsis 2: De una forma clara y lúcida, el au